In [ ]:
# ============================================================
# Cell 1: Leakage-Proof Temporal Feature Extraction & Split
# (src/feature_engineering/pipeline_builder.py)
# ============================================================
import os, sys
from pathlib import Path

PROJECT = "/kaggle/working/CyberShield-BigData"
SOURCE = "/kaggle/input/datasets/mennatullahbadawy/cybershield-bigdata-project"

# على Kaggle: انسخ المشروع المرفوع — محلياً: استخدم مجلد الريبو نفسه
if os.path.exists("/kaggle/working"):
    import shutil
    if os.path.exists(PROJECT):
        shutil.rmtree(PROJECT)
    shutil.copytree(SOURCE, PROJECT)
    os.chdir(PROJECT)
else:
    PROJECT = str(Path.cwd())

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

from src.feature_engineering.pipeline_builder import FeaturePipelineBuilder

RAW = os.path.join(PROJECT, "data/feature_store/nids_features_latest")
meta = FeaturePipelineBuilder(store_dir=RAW).build_features()
print("✅ Cell 1 complete successfully.")


In [ ]:
# ============================================================
# Cell 2: Target Sweet-Spot Benchmark (Precision >= 92% & Recall >= 90%)
# (src/orchestrator/pipeline_orchestrator.py)
# ============================================================
import os
from src.orchestrator.pipeline_orchestrator import EndToEndPipelineOrchestrator

OUT = "/kaggle/working/benchmark_results" if os.path.exists("/kaggle/working") else "./benchmark_results"

bench = EndToEndPipelineOrchestrator(store_dir=RAW, out_dir=OUT).run_training_pipeline()

results = bench["results"]
df = bench["df"]
val_predictions = bench["val_predictions"]
test_predictions = bench["test_predictions"]
models = bench["models"]
feature_scaler = bench["feature_scaler"]
X_train_t, y_train_t = bench["X_train_t"], bench["y_train_t"]
X_tr_tab, y_train = bench["X_tr_tab"], bench["y_train"]
y_val, y_test = bench["y_val"], bench["y_test"]
device = bench["device"]
print("✅ Benchmark complete!")


In [ ]:
# ============================================================
# Cell 3: Robust Generalization & IEEE/ACM Benchmark Diagnostic Suite
# (src/evaluation/model_validator.py)
# ============================================================
from src.evaluation.model_validator import ModelValidator

df_diag = ModelValidator.validate_champion_model(
    models=models,
    test_predictions=test_predictions,
    results=results,
    X_tr_tab=X_tr_tab,
    X_train_t=X_train_t,
    y_train_t=y_train_t,
    y_test=y_test,
    out_dir=OUT,
    device=device,
)


In [ ]:
# ============================================================
# Cell 4: Publication-Quality Visualization & Confusion Matrix Suite
# (src/evaluation/figures.py)
# ============================================================
from src.evaluation.figures import generate_all_figures

fig_paths = generate_all_figures(
    test_predictions=test_predictions,
    results=results,
    y_test=y_test,
    df=df,
    out_dir=OUT,
)


In [ ]:
# ============================================================
# Cell 5: Professional PDF Report + Full API
# (serving/api + src/monitoring + src/genai_reporting)
# ============================================================
import os
import numpy as np
import torch
from sklearn.preprocessing import RobustScaler

from src.models.deep_learning_models import MambaNIDS
from src.feature_store.transformations import FeatureStoreManager
from src.genai_reporting.mitre_mapping import THREAT_KB, HybridThreatRetriever
from src.genai_reporting.report_generator import IncidentReportGenerator
from src.monitoring.monitoring_engine import MLOpsMonitoringEngine
from serving.api.main import create_app, serve_in_background

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
store = FeatureStoreManager(RAW)

X_chk = np.load(os.path.join(RAW, "X_train.npy"), mmap_mode='r')
API_NF, API_SL = X_chk.shape[2], X_chk.shape[1]
print(f"⚡ Input: ({API_SL}, {API_NF})")

# استخدام موديل Mamba المُدرّب من البنش مارك (إصلاح: النوت بوك الأصلية كانت تهمله وتستخدم جديداً!)
mamba = models.get("MAMBA_SSM")
if mamba is None:
    mamba = MambaNIDS(input_dim=API_NF, d_model=64, d_state=16,
                      num_layers=2, num_classes=2, dropout=0.2).to(device)
    mpath = os.path.join(OUT, "mamba_ssm_best.pt")
    if os.path.exists(mpath):
        ckpt = torch.load(mpath, map_location=device, weights_only=False)
        try:
            mamba.load_state_dict(ckpt['model_state_dict'])
            print("✅ Model loaded")
        except Exception:
            print("⚠️ Fresh model")
else:
    print("✅ Using trained MAMBA_SSM from benchmark")
mamba.eval()

scaler_api = RobustScaler(quantile_range=(5.0, 95.0))
scaler_api.fit(np.load(os.path.join(RAW, "X_train.npy")).reshape(-1, API_NF))

FEATURE_NAMES = store.load_metadata().get("feature_names") or []
while len(FEATURE_NAMES) < API_NF:
    FEATURE_NAMES.append(f"Feature_{len(FEATURE_NAMES)}")

retriever = HybridThreatRetriever()
engine = MLOpsMonitoringEngine(mamba, scaler_api, FEATURE_NAMES, retriever, device)

# ── Batch audit: 50 attacks + 50 benign ──
print("\n🧪 Running batch inference...")
X_test_api = np.load(os.path.join(RAW, "X_test.npy"))
y_test_api = np.load(os.path.join(RAW, "y_test.npy"))
audit = engine.run_batch_audit(X_test_api, y_test_api)
tp, fp, tn, fn, roc = audit["tp"], audit["fp"], audit["tn"], audit["fn"], audit["roc"]
cm, xai_features = audit["cm"], audit["xai_features"]
threat_types_detected = audit["threat_types_detected"]

# ── FastAPI ──
ctx = {
    "mamba": mamba, "scaler": scaler_api, "feature_names": FEATURE_NAMES,
    "api_sl": API_SL, "api_nf": API_NF, "device": device,
    "retriever": retriever, "threat_kb": THREAT_KB,
    "threat_types_detected": threat_types_detected,
    "batch_stats": {"tp": tp, "fp": fp, "tn": tn, "fn": fn, "roc": roc},
    "inc_cnt": [1],
}
app = create_app(ctx)
PORT = serve_in_background(app)

# ── PDF Report ──
PDF_PATH = "/kaggle/working/CyberShield_SOC_Report.pdf" if os.path.exists("/kaggle/working") else "./CyberShield_SOC_Report.pdf"
print("\n📄 Generating Professional PDF Report...")
IncidentReportGenerator().create_soc_report(
    pdf_path=PDF_PATH, tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
    roc=float(roc), n_samples=audit["n_samples"], cm=cm,
    xai_features=xai_features, threat_types_detected=threat_types_detected,
    threat_kb=THREAT_KB, port=PORT, results=results,
)
print(f"   Threat Types: {len(threat_types_detected)}")
print(f"   API: http://localhost:{PORT}")


In [ ]:
# ============================================================
# Cell 6: Bot dependencies
# ============================================================
!pip install -q reportlab python-telegram-bot nest_asyncio requests


In [ ]:
# ============================================================
# Cell 7: SOC Telegram Live Monitor
# (src/monitoring/monitoring_engine.py)
# ============================================================
import os
import numpy as np
import nest_asyncio
nest_asyncio.apply()

X_test_raw = np.load(os.path.join(RAW, "X_test.npy"))
y_test_bot = np.load(os.path.join(RAW, "y_test.npy"))

# ضع التوكن في Environment (Kaggle Secrets) وليس في الكود:
# os.environ["CYBERSHIELD_BOT_TOKEN"] = "..."
# os.environ["CYBERSHIELD_ADMIN_CHAT_ID"] = "..."
import importlib
import src.monitoring.alert_manager as am
importlib.reload(am)
engine.alerts = am.AlertManager()

await engine.run_production_audit(
    X_test_raw=X_test_raw,
    y_test=y_test_bot,
    n_demo_attacks=2,
    poll_seconds=120,
)
